In [168]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [169]:
diabetes_data = pd.read_csv('data/diabetes_data.csv', sep=',')
diabetes_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Gender
0,6,98,58,33,190,34.0,0.430,43,0,Female
1,2,112,75,32,0,35.7,0.148,21,0,Female
2,2,108,64,0,0,30.8,0.158,21,0,Female
3,8,107,80,0,0,24.6,0.856,34,0,Female
4,7,136,90,0,0,29.9,0.210,50,0,Female


In [170]:
dupl_columns = list(diabetes_data.columns)

mask = diabetes_data.duplicated(subset=dupl_columns)
diabetes_duplicates = diabetes_data[mask]
print(f'Число найденных дубликатов: {diabetes_data.shape[0]}')

Число найденных дубликатов: 778


In [171]:
diabetes_dedupped = diabetes_data.drop_duplicates(subset=dupl_columns)
print(f'Результирующее число записей: {diabetes_dedupped.shape[0]}')

Результирующее число записей: 768


In [172]:
#список неинформативных признаков
low_information_cols = [] 

#цикл по всем столбцам
for col in diabetes_data.columns:
    #наибольшая относительная частота в признаке
    top_freq = diabetes_data[col].value_counts(normalize=True).max()
    #доля уникальных значений от размера признака
    nunique_ratio = diabetes_data[col].nunique() / diabetes_data[col].count()
    # сравниваем наибольшую частоту с порогом
    if top_freq > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(top_freq*100, 2)}% одинаковых значений')
    # сравниваем долю уникальных значений с порогом
    if nunique_ratio > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(nunique_ratio*100, 2)}% уникальных значений')

Gender: 100.0% одинаковых значений


In [173]:
information_diabetes_data = diabetes_data.drop(low_information_cols, axis=1)
print(f'Результирующее число признаков: {information_diabetes_data.shape[1]}')

Результирующее число признаков: 9


In [174]:
display(information_diabetes_data.isnull().tail())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
773,False,False,False,False,False,False,False,False,False
774,False,False,False,False,False,False,False,False,False
775,False,False,False,False,False,False,False,False,False
776,False,False,False,False,False,False,False,False,False
777,False,False,False,False,False,False,False,False,False


In [175]:
cols_null_percent = information_diabetes_data.isnull().mean() * 100
cols_with_null = cols_null_percent[cols_null_percent>0].sort_values(ascending=False)
display(cols_with_null)

Series([], dtype: float64)

In [176]:
def nulls_drop(data_f):
    diab_list_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    for col in diab_list_cols:
        data_f[col] = data_f[col].apply(lambda x: np.nan if x == 0 else x)
    return data_f

nulls_drop(information_diabetes_data)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,98.0,58.0,33.0,190.0,34.0,0.430,43,0
1,2,112.0,75.0,32.0,NaN,35.7,0.148,21,0
2,2,108.0,64.0,NaN,NaN,30.8,0.158,21,0
3,8,107.0,80.0,NaN,NaN,24.6,0.856,34,0
4,7,136.0,90.0,NaN,NaN,29.9,0.210,50,0
...,...,...,...,...,...,...,...,...,...
773,6,103.0,72.0,32.0,190.0,37.7,0.324,55,0
774,1,71.0,48.0,18.0,76.0,20.4,0.323,22,0
775,0,117.0,NaN,NaN,NaN,33.8,0.932,44,0
776,4,154.0,72.0,29.0,126.0,31.3,0.338,37,0


In [177]:
information_diabetes_data.isnull().mean().round(2).sort_values(ascending=False)

Insulin                     0.49
SkinThickness               0.30
BloodPressure               0.05
BMI                         0.01
Glucose                     0.01
Pregnancies                 0.00
DiabetesPedigreeFunction    0.00
Age                         0.00
Outcome                     0.00
dtype: float64

In [178]:
#создаем копию исходной таблицы
drop_data_diabet = information_diabetes_data.copy()
#задаем минимальный порог: вычисляем 70% от числа строк
thresh = drop_data_diabet.shape[0]*0.7
#удаляем столбцы, в которых более 30% (100-70) пропусков
drop_data_diabet = drop_data_diabet.dropna(thresh=thresh, axis=1)
#удаляем строки, в которых хотя бы 2 пропуска
m = drop_data_diabet.shape[1]
drop_data_diabet = drop_data_diabet.dropna(thresh=m-2, axis=0)
#отображаем результирующую долю пропусков
drop_data_diabet.isnull().mean()

Pregnancies                 0.000000
Glucose                     0.006485
BloodPressure               0.037613
SkinThickness               0.291829
BMI                         0.005188
DiabetesPedigreeFunction    0.000000
Age                         0.000000
Outcome                     0.000000
dtype: float64

In [179]:
drop_data_diabet.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,98.0,58.0,33.0,34.0,0.430,43,0
1,2,112.0,75.0,32.0,35.7,0.148,21,0
2,2,108.0,64.0,NaN,30.8,0.158,21,0
3,8,107.0,80.0,NaN,24.6,0.856,34,0
4,7,136.0,90.0,NaN,29.9,0.210,50,0


In [180]:
drop_data_diabet.tail()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome
773,6,103.0,72.0,32.0,37.7,0.324,55,0
774,1,71.0,48.0,18.0,20.4,0.323,22,0
775,0,117.0,NaN,NaN,33.8,0.932,44,0
776,4,154.0,72.0,29.0,31.3,0.338,37,0
777,5,147.0,78.0,NaN,33.7,0.218,65,0


In [181]:
drop_data_diabet.info()

<class 'pandas.core.frame.DataFrame'>
Index: 771 entries, 0 to 777
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               771 non-null    int64  
 1   Glucose                   766 non-null    float64
 2   BloodPressure             742 non-null    float64
 3   SkinThickness             546 non-null    float64
 4   BMI                       767 non-null    float64
 5   DiabetesPedigreeFunction  771 non-null    float64
 6   Age                       771 non-null    int64  
 7   Outcome                   771 non-null    int64  
dtypes: float64(5), int64(3)
memory usage: 54.2 KB


In [182]:
#отображаем результирующую долю пропусков
drop_data_diabet.isnull().mean().round(2).sort_values(ascending=False)

SkinThickness               0.29
BloodPressure               0.04
Glucose                     0.01
BMI                         0.01
Pregnancies                 0.00
DiabetesPedigreeFunction    0.00
Age                         0.00
Outcome                     0.00
dtype: float64

In [183]:
#В оставшихся записях замените пропуски на медиану. Чему равно среднее значение в столбце SkinThickness? Ответ округлите до десятых.
null_data = drop_data_diabet.isnull().sum()
cols = null_data[null_data>0].index
for col in cols:
    drop_data_diabet[col] = drop_data_diabet[col].fillna(drop_data_diabet[col].median())
print(drop_data_diabet['SkinThickness'].mean().round(1))

29.1


In [184]:
def outliers_iqr(data, feature):
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 1.5)
    upper_bound = quartile_3 + (iqr * 1.5)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_iqr(drop_data_diabet, 'SkinThickness')
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу Тьюки: 87
Результирующее число записей: 684


In [185]:
def outliers_z_score(data, feature, log_scale=False):
    if log_scale:
        x = np.log(data[feature])
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - 3 * sigma
    upper_bound = mu + 3 * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_z_score(drop_data_diabet, 'SkinThickness')
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу Тьюки: 4
Результирующее число записей: 767


In [186]:
outliers, cleaned = outliers_iqr(drop_data_diabet, 'DiabetesPedigreeFunction')
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу Тьюки: 29
Результирующее число записей: 742


In [187]:
outliers, cleaned = outliers_z_score(drop_data_diabet, 'DiabetesPedigreeFunction', log_scale = True)
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу Тьюки: 0
Результирующее число записей: 771
